# 01 — Run the chart-review agent

This notebook answers one question: **what does the agent do, and what answer did it
return?** Analysis comes later.

## The agent in one picture

```text
task
  ↓
choose the next useful chart action
  ↓
call a chart tool → observe the result → update working state
  ↑                                      ↓
  └──────────────── repeat ───────────────┘
                                          ↓
                                    submit answer
```

Codex supplies the ReAct-style loop. ACR gives it a **chart-only boundary**. The agent
cannot use a shell, browser, or arbitrary filesystem access.

| Job | Tool | Plain meaning |
|---|---|---|
| Find | `list_documents` | See which notes exist. |
| Find | `search` | Find notes containing a term. |
| Read | `read` | Open one note. |
| Explain a consequential choice | `note_decision` | Record the question, choice, reason, alternatives, and claimed basis. |
| Judge evidence | `record_finding` | Say whether one note can establish the requested field. |
| Preserve proof | `record_evidence` | Save the exact supporting span. |
| Finish | `submit_answer` | Submit only after the evidence gate is satisfied. |

`note_decision` is a short audit explanation, **not private chain-of-thought**.

We use two instruction sets:

- **Task only:** asks for the diagnosis date and output format, but withholds the
  clinical evidence/conflict rules.
- **Task + policy:** adds explicit rules for what counts as evidence, which date wins,
  and what must be cited before submission.

Existing real-provider runs are reused by default, so simply reading this notebook
makes no paid call. Set `ACR_TUTORIAL_MODE=live` to run Luna again.


**Checked reading copy.** The saved outputs below come from completed real-provider runs. Implementation cells are omitted here for readability; open [`01_run_chart_review_experiments.ipynb`](./01_run_chart_review_experiments.ipynb) to inspect or rerun the code.


## Choose what to run

The live pilot is deliberately small: two paired synthetic cases × two instruction
sets × Luna. `SYN0001` has an early same-day physician diagnosis; `SYNX03` does not.
That single difference tests whether the agent handles ambiguous cytology correctly.


**Mode:** `reuse` · **Codex:** `codex-cli 0.150.1`

Loading the available historical examples with complete sealed metadata.

## Run

In live mode the collapsed cell below performs the closed loop:

```text
Luna chart review → local Langtrace → two Luna reconstructions
→ verifier agreement → selected analysis → Semantica
```

The reconstruction work is packaged here only so Notebooks 2 and 3 have something to
read. This notebook does not analyze it.


## What came back?

This is intentionally the only result table in Notebook 1. Synthetic gold is shown
only because these tutorial cases were designed with a known answer.


| Case | Instructions | Model | Agent answer | Synthetic gold | Match? |
|---|---|---|---|---|---|
| SYN0001 | Task only | Luna | 20230427 | 20230412 | ✗ |
| SYN0001 | Task only | Terra | 20230427 | 20230412 | ✗ |
| SYN0001 | Task only | Terra | 20230412 | 20230412 | ✓ |
| SYN0001 | Task only | Luna | 20230412 | 20230412 | ✓ |
| SYN0001 | Task + policy | Luna | 20230412 | 20230412 | ✓ |
| SYNX03 | Task + policy | Terra | 20220309 | 20220309 | ✓ |

**Read this correctly:** these are 6 historical runs with complete metadata, not a balanced accuracy experiment. 1 incomplete result directory was not treated as a run. Notebook 2 now follows one run step by step.